# Notebook 00 — Lancer le pipeline automatiquement (le "bouton")

**Objectif** : après avoir modifié un ou plusieurs paramètres dans `config.py`, exécuter
automatiquement **les notebooks nécessaires et suffisants** (parmi 02 à 07) pour que tes
modèles ET leurs portefeuilles long-short soient à jour -- sans avoir à te souvenir
toi-même de quel notebook dépend de quel paramètre, ni à relancer à la main ceux qui n'ont
pas besoin de l'être.

**Comment ça marche** : `pipeline.py` connaît la chaîne de dépendances du projet --

| Notebook | Dépend directement de (dans `config.py`) | Dépend aussi de |
|---|---|---|
| 02 — nettoyage | `ANNEE_DEBUT`, `CARACTERISTIQUES`, `SEUIL_MAX_PCT_MANQUANT_CARACTERISTIQUES` | — |
| 03 — construction panel | `SEUIL_PERCENTILE_TAILLE`, `SEUIL_PERCENTILE_LIQUIDITE` | sortie de 02 |
| 04 — régression linéaire | `PREDICTEURS`, `TYPE_FENETRE`, `ANNEES_*` | sortie de 03 |
| 05 — Elastic Net | idem 04 + `GRILLE_ALPHA_ELASTIC_NET`, `GRILLE_L1_RATIO_ELASTIC_NET`, `MAX_ITER_ELASTIC_NET` | sortie de 03 |
| 06 — LightGBM | idem 04 + `GRILLE_NUM_LEAVES_LIGHTGBM`, `GRILLE_LEARNING_RATE_LIGHTGBM`, `GRILLE_MIN_CHILD_SAMPLES_LIGHTGBM`, `N_ESTIMATORS_LIGHTGBM`, `STOPPING_ROUNDS_LIGHTGBM` | sortie de 03 |
| 07 — portefeuilles long-short | `NB_DECILES` | sortie de **04 ET 05 ET 06** |

Un notebook est "sale" (à re-exécuter) si l'un de **ses propres** paramètres a changé
depuis son dernier lancement réussi via ce notebook 00, **ou** si un notebook dont il
dépend est lui-même sale (la sale-té se propage vers l'aval de la chaîne). Exemples :
- Tu changes seulement `TYPE_FENETRE` → seuls 04, 05, 06 sont relancés, **et 07 avec eux**
  (il depend des 3) -- 02 et 03 restent intacts, ils ne dépendent pas de ce paramètre.
- Tu changes `SEUIL_PERCENTILE_TAILLE` → 03 est sale, ce qui rend 04, 05, 06 sales aussi
  **en chaîne** (leur entrée, `panel_pret_modelisation.parquet`, a changé), même si aucun
  de leurs propres paramètres n'a bougé -- et 07 en chaîne derrière eux. 02 reste intact.
- Tu changes `CARACTERISTIQUES` (l'univers candidat des 94) ou
  `SEUIL_MAX_PCT_MANQUANT_CARACTERISTIQUES` (le seuil d'exclusion des candidates trop
  incomplètes, voir notebook 02 section A.3bis) → 02 est sale, donc **tout** le reste
  l'est aussi en chaîne (`PREDICTEURS`, recalculé à partir du sous-ensemble retenu,
  change lui aussi automatiquement).
- Tu changes seulement `GRILLE_NUM_LEAVES_LIGHTGBM` → seuls 06 et 07 sont sales (07 en
  chaîne, puisqu'il depend de 06) ; 04 et 05 ne sont pas concernés.
- Tu changes seulement `NB_DECILES` → seul 07 est sale (aucun des 3 modèles n'en dépend).

⚠️ **Cas particulier : cibler un seul modèle.** '07_evaluation_portefeuilles' dépend des
**TROIS** modèles à la fois (il a besoin de leurs `resultats_*`/`predictions_*` en même
temps). Si tu retires volontairement un modèle des `CIBLES` ci-dessous (ex: pour ne
t'occuper que de LightGBM), retire aussi `'07_evaluation_portefeuilles'` de `CIBLES`,
sans quoi les 2 autres modèles seraient entraînés quand même juste pour satisfaire 07 --
relance 07 à la main séparément dans ce cas.


## 0. Import

In [1]:
import sys
sys.path.append("..")  # pour pouvoir importer config.py et pipeline.py, situes a la racine du projet
import config
import pipeline

config.assurer_dossiers()

try:
    import papermill  # noqa: F401
    print("papermill est installe -- pret a executer le pipeline.")
except ImportError:
    print("⚠️  papermill n'est PAS installe dans ce kernel.")
    print("    Lance dans un terminal (ou une cellule avec !) :  pip install papermill")
    print("    (ou 'pip install papermill --break-system-packages' selon ton systeme), puis redemarre ce kernel.")

papermill est installe -- pret a executer le pipeline.


## 1. Configuration de ce lancement

Modifie ces quelques variables si besoin, puis exécute la cellule suivante -- le **plan
d'exécution est toujours affiché en premier**, avant que quoi que ce soit ne soit
réellement lancé, pour que tu puisses vérifier avant de lancer un calcul potentiellement
long (LightGBM notamment).

In [ ]:
#si run notebook manuellement, on peut forcer la mise a jour de certains notebooks en les listant ici.
"pipeline.marquer_a_jour('02_nettoyage_donnees', '03_construction_panel', '04_modele_lineaire', '05_modele_elastic_net', '06_modele_lightgbm', '07_evaluation_portefeuilles')"

# Quels notebooks veux-tu avoir a jour ? Par defaut, les 3 modeles ET leurs portefeuilles
# long-short (07) -- pipeline.py determine tout seul le sous-ensemble necessaire et
# suffisant parmi 02 a 07 pour y arriver.
CIBLES = ('04_modele_lineaire', '05_modele_elastic_net', '06_modele_lightgbm', '07_evaluation_portefeuilles')

# Relancer aussi le notebook 08 (comparaison des experiences) a la fin ? Recommande : True
# (c'est sans risque, il ne fait que lire le journal + le dernier
# outputs/performance_portefeuilles.parquet ecrit par 07, jamais ré-entrainer quoi que ce soit).
INCLURE_NOTEBOOK_08 = True

# Notebooks a re-executer INCONDITIONNELLEMENT (parmi les noms de GRAPHE_DEPENDANCES,
# 02 a 07), meme si la detection automatique les jugerait a jour -- vide par defaut. Utile
# si tu as modifie un fichier de donnees a la main, ou juste pour regenerer des figures.
FORCER = []

# Nom du kernel Jupyter a utiliser (None = kernel par defaut de papermill) -- utile
# seulement si plusieurs environnements/kernels sont installes et que papermill se trompe.
KERNEL_NAME = None

# Mets True pour VOIR SEULEMENT le plan d'execution, SANS RIEN LANCER -- recommande avant
# un premier essai, ou apres avoir change beaucoup de parametres d'un coup.
APERCU_SEULEMENT = True

## 2. Le bouton

In [3]:
notebooks_lances = pipeline.executer_pipeline(
    cibles=CIBLES,
    inclure_08=INCLURE_NOTEBOOK_08,
    forcer=FORCER,
    kernel_name=KERNEL_NAME,
    executer=not APERCU_SEULEMENT,
)

Plan d'execution (d'apres les parametres ACTUELS de config.py) :
  [A EXECUTER]  02_nettoyage_donnees         -- parametres/entree modifies (ou jamais execute par pipeline.py)
  [A EXECUTER]  03_construction_panel        -- parametres/entree modifies (ou jamais execute par pipeline.py)
  [A EXECUTER]  04_modele_lineaire           -- parametres/entree modifies (ou jamais execute par pipeline.py)
  [A EXECUTER]  05_modele_elastic_net        -- parametres/entree modifies (ou jamais execute par pipeline.py)
  [A EXECUTER]  06_modele_lightgbm           -- parametres/entree modifies (ou jamais execute par pipeline.py)
  [A EXECUTER]  07_evaluation_portefeuilles  -- parametres/entree modifies (ou jamais execute par pipeline.py)
  [A EXECUTER]  08_comparaison_experiences   -- toujours relance (lecture seule, sans risque)

executer=False : rien n'a ete lance, plan affiche seulement.


## 3. Notes

- ⚠️ **Si tu lances un notebook TOI-MEME** (Run All dans Jupyter) plutôt que via ce
  notebook 00, `outputs/etat_pipeline.json` n'est PAS mis à jour (seul
  `executer_pipeline` l'écrit) -- au prochain passage ici, ce notebook sera donc
  considéré "sale" et relancé, **même s'il est déjà à jour** (pas faux, juste une perte
  de temps -- le journal des expériences, lui, ne crée jamais de doublon quoi qu'il
  arrive). Pour éviter ce gaspillage après un lancement manuel, synchronise l'état à la
  main, sans rien ré-exécuter :
  ```python
  pipeline.marquer_a_jour('04_modele_lineaire', '05_modele_elastic_net', '06_modele_lightgbm', '07_evaluation_portefeuilles')
  ```
  (ne l'utilise que juste après avoir toi-même lancé ces notebooks jusqu'au bout, sans
  erreur, avec les paramètres actuels de `config.py` -- )
- **Repartir de zéro** : supprime `outputs/etat_pipeline.json` -- au prochain lancement,
  tous les notebooks concernés seront considérés comme jamais exécutés, et donc relancés.
- **En cas d'échec** en cours de route (une cellule plante dans un notebook, ex: LightGBM
  à court de mémoire) : le pipeline s'arrête net, les notebooks déjà terminés avec succès
  restent enregistrés comme à jour (pas besoin de les relancer), et le message d'erreur
  t'indique lequel a échoué -- ouvre-le directement dans Jupyter pour voir la trace
  complète (papermill y insère une cellule d'erreur à l'endroit exact où ça a planté).
- **Verrous de fichiers (OneDrive/Google Drive/Dropbox/antivirus)** : `pipeline.py`
  minimise déjà ce risque (une seule sauvegarde par notebook, à la toute fin) et réessaie
  automatiquement 3 fois -- si l'erreur `PermissionError` persiste, ferme le notebook
  concerné ailleurs (Jupyter/VS Code) et/ou mets la synchronisation en pause le temps du
  lancement.
- **07 dépend des 3 modèles à la fois** : si tu restreins `CIBLES` (section 1) à un seul
  modèle, retire aussi `'07_evaluation_portefeuilles'` de `CIBLES`, sinon les 2 autres
  modèles seront entraînés quand même pour satisfaire 07 (voir la section 1 ci-dessus).
- Ce notebook ne fait *que* orchestrer les autres : il ne contient lui-même aucune
  logique de nettoyage, de modélisation ou d'évaluation.